# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

We are trying to predict whether a webpage's performance will grow, decline, or stay stable, so an SEO/content team can decide which pages to review first. If we miss a page that's about to decline, the team loses traffic without knowing and if we flag too many stable pages as risky, the team wastes review time. There might be multiple factors that are making a declined webpage and a simple hand rule can't guess all of them and might miss some also it will take so much time that's why we are using a ML model instead of hand written straightforward rules

In [1]:
target_result=['Growing','Declining','Stable']
model_type='Ranking'
evaluation_metric='precision@k'
print(f"Our target result will tell us the status of our webpage such as {target_result[0]}, {target_result[1]}, or {target_result[2]} ")
print(f"We will be using {model_type} method with the {evaluation_metric} as evaluation metrics")

Our target result will tell us the status of our webpage such as Growing, Declining, or Stable 
We will be using Ranking method with the precision@k as evaluation metrics


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

For this project we have used `FlyRank/internship-warehouse's` mid panel `month march 2026` data from `fact_content_daily_performance` folder. After initial inspection we found that we have nearly `9.9M` rows and `30` columns in that data.
We found that almost 31% data in `ga4_data_available` is missing, exactly the rows where `client_has_ga4` is `False`. It shows that almost `31%` client don't have ga4(on-page behavior), also GSC data(search visibility) has its separate missingness in `gsc_avg_position ` column, almost `6.2M` rows has missing values which makes almost `63.30%` missing data from all rows, way more than `ga4_data_available`. 

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
df_march = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/",storage_options={"token": hf_token})
df_april=pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/",storage_options={"token": hf_token})
print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())
print(df_march.columns.tolist())
df_march.head()

(9841378, 30)
2026-03-01 2026-03-31
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print(df_april.shape)
print(df_april['report_date'].min(), df_april['report_date'].max())
print(df_april.columns.to_list())
df_april.head()

(10424730, 30)
2026-04-01 2026-04-30
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-04-01,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,True,False,True,None,9,0,493,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-01,client_62f4a7e64f5e0096,content_13a8105125458098,True,False,True,None,1,0,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-01,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-01,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,True,False,True,None,1,0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-01,client_62f4a7e64f5e0096,content_ddbfb1907979759a,True,False,False,None,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df_april.isna().sum()/len(df_april)*100

report_date                  0.000000
client_hash_id               0.000000
content_hash_id              0.000000
client_has_gsc               0.000000
client_has_ga4               0.000000
gsc_data_available           0.000000
ga4_data_available          21.257634
gsc_impressions              0.000000
gsc_clicks                   0.000000
gsc_sum_position             0.000000
gsc_avg_position            62.578897
ga4_pageviews               21.257634
ga4_sessions                21.257634
ga4_users                   21.257634
ga4_engaged_sessions        21.257634
ga4_total_engagement_sec    21.257634
sessions_organic            21.257634
sessions_direct             21.257634
sessions_referral           21.257634
sessions_social             21.257634
sessions_paid               21.257634
sessions_ai                 21.257634
ai_chatgpt                  21.257634
ai_perplexity               21.257634
ai_gemini                   21.257634
ai_copilot                  21.257634
ai_claude   

In [3]:
df_march.isna().sum()/len(df_march)*100

report_date                  0.000000
client_hash_id               0.000000
content_hash_id              0.000000
client_has_gsc               0.000000
client_has_ga4               0.000000
gsc_data_available           0.000000
ga4_data_available          30.673967
gsc_impressions              0.000000
gsc_clicks                   0.000000
gsc_sum_position             0.000000
gsc_avg_position            63.307364
ga4_pageviews               30.673967
ga4_sessions                30.673967
ga4_users                   30.673967
ga4_engaged_sessions        30.673967
ga4_total_engagement_sec    30.673967
sessions_organic            30.673967
sessions_direct             30.673967
sessions_referral           30.673967
sessions_social             30.673967
sessions_paid               30.673967
sessions_ai                 30.673967
ai_chatgpt                  30.673967
ai_perplexity               30.673967
ai_gemini                   30.673967
ai_copilot                  30.673967
ai_claude   

In [4]:
df_march['client_has_ga4'].value_counts()

client_has_ga4
True     6822637
False    3018741
Name: count, dtype: int64

In [5]:
df_march['ai_chatgpt'].value_counts()

ai_chatgpt
0.0     6819460
1.0        1941
2.0         882
3.0         188
4.0          84
5.0          40
6.0          22
7.0           7
8.0           5
11.0          2
12.0          2
14.0          1
17.0          1
28.0          1
24.0          1
Name: count, dtype: int64

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

We will use following threshold to call something "Growing", "Stable" , and "Declining":
- Growth = above +15%
- Declining = below -10%
- Stable= Between -10% and +15%

We made it asymmetric intentionaly because declining pages needs to be catch earlier before it causes some major problems. The cost of late detection of declining pages is more than that of observing growth patterns.

In [ ]:
# Calculating Sum Of GSC Clicks Per Page From Both Months
march_sum=df_march.groupby('content_hash_id')['gsc_clicks'].sum()
print("March Month")
print(march_sum)
print("\nApril Month")
april_sum=df_april.groupby('content_hash_id')['gsc_clicks'].sum()
print(april_sum)

# Converting Both Results To Dataframes



March Month
content_hash_id
content_000005d4ced12088     0
content_00001e488b74b799     0
content_00007bd2985b77c3     0
content_00008950670cb6b5     0
content_0000a348850eb1fc     0
                            ..
content_ffffbc148e416f89     0
content_ffffc282ab2cbe62     0
content_ffffc58385523096    37
content_ffffe701567e982c     0
content_fffff09da8a25da6     0
Name: gsc_clicks, Length: 331437, dtype: int64

April Month
content_hash_id
content_000005d4ced12088     0
content_00001e488b74b799     0
content_00007bd2985b77c3     0
content_00008950670cb6b5     0
content_0000a348850eb1fc     0
                            ..
content_ffffbc148e416f89     0
content_ffffc282ab2cbe62     0
content_ffffc58385523096    22
content_ffffe701567e982c     0
content_fffff09da8a25da6     2
Name: gsc_clicks, Length: 362172, dtype: int64


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
